# JFMO Dataset Creation

Download and process datasets for EN-RU language identification model.

## Datasets
- **RU titles**: [Movie plots from Wikipedia in Russian](https://www.kaggle.com/datasets/maksimpotorochin/movie-plots-from-wikipedia-in-russian) + [Kion](https://ods.ai/competitions/competition-recsys-21/data) + [IMDb](https://developer.imdb.com/non-commercial-datasets/)
- **EN titles**: [Full TMDB Movies Dataset 2024 (1M Movies)](https://www.kaggle.com/datasets/asaniczka/tmdb-movies-dataset-2023-930k-movies) + [IMDb](https://developer.imdb.com/non-commercial-datasets/)

## Imports

In [ ]:
!pip install -q transliterate

In [ ]:
import os
import transliterate

from tqdm import tqdm
import pandas as pd

## Setup

In [ ]:
!rm -rf ./data sample_data

os.environ['KAGGLE_USERNAME'] = ''
os.environ['KAGGLE_KEY'] = ''

## Download Datasets

### RU Dataset 1: Movie plots from Wikipedia in Russian

In [ ]:
!kaggle datasets download maksimpotorochin/movie-plots-from-wikipedia-in-russian
!unzip -q movie-plots-from-wikipedia-in-russian.zip -d ./data
!rm -rf movie-plots-from-wikipedia-in-russian.zip

### RU Dataset 2: Kion

In [ ]:
!wget https://storage.yandexcloud.net/datasouls-ods/materials/f90231b6/items.csv -P ./data

### IMDB RU + EN

In [ ]:
!wget https://datasets.imdbws.com/title.akas.tsv.gz -P ./data
!gunzip ./data/title.akas.tsv.gz

### EN Dataset: TMDB

In [ ]:
!kaggle datasets download 'asaniczka/tmdb-movies-dataset-2023-930k-movies'
!unzip -q 'tmdb-movies-dataset-2023-930k-movies'.zip -d ./data
!rm -rf 'tmdb-movies-dataset-2023-930k-movies'.zip

## Read Datasets

### RU Dataset 1

In [ ]:
df_ru1 = pd.read_csv('data/films_data.csv')
print("RU Dataset 1 (films_data.csv):")
print(f"  Columns: {df_ru1.columns.tolist()}")
print(f"  Rows: {len(df_ru1)}")
print(df_ru1.head())

### RU Dataset 2

In [ ]:
df_ru2 = pd.read_csv('data/items.csv', encoding='utf-8')
print("\nRU Dataset 2 (items.csv):")
print(f"  Columns: {df_ru2.columns.tolist()}")
print(f"  Rows: {len(df_ru2)}")
print(df_ru2.head())

### IMDB (RU + EN)

In [ ]:
chunks_ru = []
chunks_en = []
chunk_size = 100000

for chunk in tqdm(
    pd.read_csv('data/title.akas.tsv', sep='\t', chunksize=chunk_size, low_memory=False),
    desc="Processing IMDB chunks"
):
    chunk_ru = chunk[
        (chunk['region'] == 'RU') |
        (chunk['language'] == 'ru')
    ]
    if len(chunk_ru) > 0:
        chunks_ru.append(chunk_ru)

    chunk_en = chunk[
        (chunk['region'] == 'US') |
        (chunk['region'] == 'GB') |
        (chunk['language'] == 'en')
    ]
    if len(chunk_en) > 0:
        chunks_en.append(chunk_en)

### EN Dataset: TMDB

In [ ]:
df_en = pd.read_csv('data/TMDB_movie_dataset_v11.csv')
print("\nEN Dataset (TMDB):")
print(f"  Columns: {df_en.columns.tolist()}")
print(f"  Rows: {len(df_en)}")
print(df_en.head())

## Extract Titles

### RU Titles

In [ ]:
ru_titles_1 = df_ru1['title'].dropna().tolist()
ru_types_1 = df_ru1['type'].dropna().tolist()
print(f"RU titles from films_data.csv: {len(ru_titles_1)}")

ru_titles_2 = df_ru2['title'].dropna().tolist()
ru_types_2 = df_ru2['content_type'].dropna().tolist()
print(f"RU titles from items.csv: {len(ru_titles_2)}")

df_imdb_ru = pd.concat(chunks_ru, ignore_index=True) if chunks_ru else pd.DataFrame()
ru_titles_3 = df_imdb_ru['title'].dropna().tolist()
ru_types_3 = [None] * len(ru_titles_3)
print(f"RU titles from IMDB: {len(ru_titles_3)}")

### EN Titles

In [ ]:
en_titles_1 = df_en['title'].dropna().tolist()
en_types_1 = ['movie'] * len(en_titles_1)
print(f"EN titles from TMDB: {len(en_titles_1)}")

df_imdb_en = pd.concat(chunks_en, ignore_index=True) if chunks_en else pd.DataFrame()
en_titles_2 = df_imdb_en['title'].dropna().tolist()
en_types_2 = [None] * len(en_titles_2)
print(f"EN titles from IMDB: {len(en_titles_2)}")

## Normalize & Deduplicate

In [ ]:
def normalize_type(type_val):
    if type_val is None:
        return None
    type_str = str(type_val).lower().strip()
    if 'film' in type_str or 'movie' in type_str:
        return 'movie'
    elif 'series' in type_str:
        return 'tv series'
    return type_val

### RU Titles (deduplicate)

In [ ]:
all_ru_titles = ru_titles_1 + ru_titles_2 + ru_titles_3
all_ru_types = ru_types_1 + ru_types_2 + ru_types_3

ru_titles_unique = []
ru_types_unique = []
seen = set()

for title, type_val in zip(all_ru_titles, all_ru_types):
    title_lower = str(title).lower().strip()
    if title_lower not in seen:
        seen.add(title_lower)
        ru_titles_unique.append(title)
        ru_types_unique.append(normalize_type(type_val))

print(f"Unique RU titles: {len(ru_titles_unique)}")

### EN Titles (deduplicate)

In [ ]:
all_en_titles = en_titles_1 + en_titles_2
all_en_types = en_types_1 + en_types_2

en_titles_unique = []
en_types_unique = []
seen_en = set()

for title, type_val in zip(all_en_titles, all_en_types):
    title_lower = str(title).lower().strip()
    if title_lower not in seen_en:
        seen_en.add(title_lower)
        en_titles_unique.append(title)
        en_types_unique.append(type_val)

print(f"Unique EN titles: {len(en_titles_unique)}")

## Transliterate RU Titles

In [ ]:
print(f"Transliterating {len(ru_titles_unique)} titles...")

ru_titles_translit = []
errors_ru = 0

for text in tqdm(ru_titles_unique, desc="Transliterating"):
    try:
        translit = transliterate.translit(str(text), 'ru', reversed=True)
        ru_titles_translit.append(translit)
    except Exception as e:
        errors_ru += 1
        ru_titles_translit.append(None)

print(f"\nTransliteration errors: {errors_ru}")

## Create Final Datasets

### RU Dataset

In [ ]:
df_ru_final = pd.DataFrame({
    'title': ru_titles_unique[:len(ru_titles_translit)],
    'language': 'ru',
    'title_translit': ru_titles_translit,
    'type': ru_types_unique[:len(ru_titles_translit)]
})

df_ru_final = df_ru_final[df_ru_final['title_translit'].notna()]
print(f"RU final (after removing failed): {len(df_ru_final)} rows")
print(df_ru_final.head())

### EN Dataset

In [ ]:
df_en_final = pd.DataFrame({
    'title': en_titles_unique,
    'language': 'en',
    'title_translit': None,
    'type': en_types_unique
})

print(f"EN final: {len(df_en_final)} rows")
print(df_en_final.head())

### Combined Dataset

In [ ]:
train_df = pd.concat([df_ru_final, df_en_final], ignore_index=True)

print(f"\nTrain dataset combined: {len(train_df)} rows")
print(f"  - RU: {len(df_ru_final)} rows")
print(f"  - EN: {len(df_en_final)} rows")
print(f"\nColumns: {train_df.columns.tolist()}")
print("\nSample rows:")
print(train_df.head(10))

## Save Dataset

In [ ]:
train_df.to_csv('media_transliterated.csv', index=False, encoding='utf-8')
print(f"\n💾 Dataset saved: media_transliterated.csv ({len(train_df)} rows)")